<h1 style="
    color:#FFFFFF;
    background:linear-gradient(90deg,#0F766E,#0284C7);
    text-align:center;
    font-weight:bold;
    padding:18px 12px;
    border-radius:10px;
    margin-bottom:7px;">
    Week 9 — Day 2: Serving the Model with FastAPI
</h1>

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">
    From Saved IMDb Artifacts to a REST Prediction API
</h3>

<div style="border-left:6px solid #FB7185;background-color:#FFF1F2;padding:12px 16px;margin:16px 0;border-radius:6px;color:#7F1D1D;">
<b>Week 9 — Model Deployment • Sprint 4 • Day 2</b><br>
Project used today: <b>IMDb Sentiment Analysis</b> from Week 8.<br>
Day 1 result: a serialized TF-IDF vectorizer, Logistic Regression model, and reusable text preprocessing pipeline.
</div>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p><b>Main question:</b></p>
<div style="font-size:1.12em;font-weight:bold;text-align:center;background-color:#FFFBEB;border:1px solid #FBBF24;padding:16px;border-radius:8px;margin:14px 0;color:#111827;">
How do we expose the saved sentiment model through an API so another program can send a movie review and receive a prediction?
</div>
</div>

<a id="toc"></a>
<h2 style="color:#92400E; background-color:#FBBF24; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">Table of Contents</h2>
<div style="border:1px solid #CBD5E1; padding:16px 25px; border-radius:8px; background-color:#FFFFFF; color:#111827;">
<ul style="font-weight:bold; line-height:1.95; color:#1F2937;">
<li><a href="#section0">0. Setup — Files, Imports & Environment</a></li>
<li><a href="#section1">1. Day 2 Learning Map</a></li>
<li><a href="#section2">2. What FastAPI Does</a></li>
<li><a href="#section3">3. REST API, Endpoint & POST</a></li>
<li><a href="#section4">4. Load the Saved Deployment Artifacts</a></li>
<li><a href="#section5">5. Pydantic Input Validation</a></li>
<li><a href="#section6">6. Create the FastAPI Application</a></li>
<li><a href="#section7">7. Build POST /predict</a></li>
<li><a href="#section8">8. Test the API Inside the Notebook</a></li>
<li><a href="#section9">9. Create main.py</a></li>
<li><a href="#section10">10. Run the API with Uvicorn</a></li>
<li><a href="#section11">11. Test with FastAPI /docs</a></li>
<li><a href="#section12">12. Validate Invalid Input</a></li>
<li><a href="#section13">13. Day 2 Definition of Done</a></li>
<li><a href="#section14">14. What I Learned Today</a></li>
</ul>
</div>

<a id="section0"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">0. Setup — Files, Imports & Environment</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">0.1 Expected folder structure</h3>


<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">The Main Folder/
├── Week 8/
│   └── Day 2/
│       ├── tfidf_vectorizer.joblib
│       └── sentiment_model.joblib
└── Week 9/
    ├── Day 1/
    │   └── preprocessing.py
    └── Day 2/
        ├── Day2.ipynb
        ├── preprocessing.py
        ├── tfidf_vectorizer.joblib
        ├── sentiment_model.joblib
        └── main.py</div>

<p>Day 2 serves the same trained pipeline prepared in Day 1. We do not retrain the model.</p>

<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Important:</b> Run this notebook from <code>Week 9/Day 2</code>. Keep the model, vectorizer, and <code>preprocessing.py</code> beside the notebook before starting Uvicorn.</div>

</div>

In [1]:
from pathlib import Path
import shutil
import sys

DAY2_DIR = Path.cwd().resolve()

<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;">
<b>Day 2 rule:</b> The API must load the saved model and saved TF-IDF vectorizer. It should never retrain them when a request arrives.
</div>

In [2]:
# Try to bring the Day 1 / Week 8 deployment files into Day 2 automatically.
model_dest = DAY2_DIR / "sentiment_model.joblib"
vectorizer_dest = DAY2_DIR / "tfidf_vectorizer.joblib"
preprocessing_dest = DAY2_DIR / "preprocessing.py"

week8_day2 = DAY2_DIR.parent.parent / "Week 8" / "Day 2"
week9_day1 = DAY2_DIR.parent / "Day 1"

sources = {
    model_dest: week8_day2 / "sentiment_model.joblib",
    vectorizer_dest: week8_day2 / "tfidf_vectorizer.joblib",
    preprocessing_dest: week9_day1 / "preprocessing.py",
}

for destination, source in sources.items():
    if destination.exists():
        print(f" Already exists: {destination.name}")
    elif source.exists():
        shutil.copy2(source, destination)
        print(f" Copied: {source.name} → Day 2")
    else:
        print(f" Could not find: {source}")

 Already exists: sentiment_model.joblib
 Already exists: tfidf_vectorizer.joblib
 Already exists: preprocessing.py


In [3]:
required_files = [
    DAY2_DIR / "sentiment_model.joblib",
    DAY2_DIR / "tfidf_vectorizer.joblib",
    DAY2_DIR / "preprocessing.py",
]

for file in required_files:
    print(f"{'' if file.exists() else ''} {file.name}")

 sentiment_model.joblib
 tfidf_vectorizer.joblib
 preprocessing.py


<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">0.2 Required packages</h3>
<p>Day 2 uses FastAPI, Pydantic, Uvicorn, joblib, and the libraries already required by the sentiment preprocessing pipeline.</p>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">fastapi
uvicorn
pydantic
joblib
httpx</div>
</div>

In [4]:
# Run only if needed:
# %pip install fastapi uvicorn pydantic joblib httpx

<a id="section1"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">1. Day 2 Learning Map</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>Day 1 prepared the deployment artifacts. Day 2 turns those artifacts into a service that other applications can call.</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Day 1
Saved preprocessing
      ↓
Saved TF-IDF vectorizer
      ↓
Saved Logistic Regression model

Week 9 — Day 2
      ↓
FastAPI application
      ↓
Pydantic validation
      ↓
POST /predict
      ↓
Same preprocessing
      ↓
TF-IDF transform
      ↓
Model prediction
      ↓
JSON response</div>

<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Today</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Why</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">FastAPI</td><td style="padding:9px;border:1px solid #CBD5E1;">Expose the trained model as a REST service.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Pydantic</td><td style="padding:9px;border:1px solid #CBD5E1;">Validate input before it reaches the model.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">POST /predict</td><td style="padding:9px;border:1px solid #CBD5E1;">Receive a review and return a prediction.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">/docs</td><td style="padding:9px;border:1px solid #CBD5E1;">Test the endpoint interactively in the browser.</td></tr>
</table>
</div>

<a id="section2"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">2. What FastAPI Does</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">2.1 Definition</h3>

<p><b>FastAPI</b> is a Python framework for building REST APIs. In our project, it creates a web service around the sentiment model.</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Without FastAPI:
Notebook → model.predict(...)

With FastAPI:
Website / App
      ↓
HTTP request
      ↓
FastAPI
      ↓
Sentiment model
      ↓
HTTP response</div>


<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">2.2 Why it matters</h3>

<p>A website, mobile app, or another backend does not need to know how TF-IDF or Logistic Regression work. It only needs to send a valid request to the API.</p>
</div>

<a id="section3"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">3. REST API, Endpoint & POST</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">3.1 REST API</h3>

<p>A REST API exposes URL paths that programs can call over HTTP.</p>

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">3.2 Endpoint</h3>

<p>An <b>endpoint</b> is a specific URL path with a specific job. Our main endpoint is:</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">POST /predict</div>


<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">3.3 POST request</h3>

<p>We use POST because the client sends review data to the server.</p>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">{
  "review": "This movie was amazing"
}</div>
</div>

<a id="section4"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">4. Load the Saved Deployment Artifacts</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">4.1 Load instead of retrain</h3>

<p>The API loads the objects produced earlier. It must not call <code>fit()</code>.</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">sentiment_model.joblib
        ↓
    joblib.load()
        ↓
trained Logistic Regression

tfidf_vectorizer.joblib
        ↓
    joblib.load()
        ↓
trained TF-IDF vectorizer</div>


<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Important:</b> During serving, use <code>vectorizer.transform()</code>, not <code>fit_transform()</code>. The vectorizer already learned its vocabulary during training.</div>

</div>

In [5]:
import joblib

vectorizer = joblib.load("tfidf_vectorizer.joblib")
model = joblib.load("sentiment_model.joblib")

print(type(vectorizer))
print(type(model))
print("Saved deployment artifacts loaded.")

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
<class 'sklearn.linear_model._logistic.LogisticRegression'>
Saved deployment artifacts loaded.


In [6]:
from preprocessing import preprocess_to_string

sample_review = "I didn't like this movie!!! It was terrible."
clean_review = preprocess_to_string(sample_review)

print("Original review:", sample_review)
print("Clean review:", clean_review)

Original review: I didn't like this movie!!! It was terrible.
Clean review: not like movie terrible


In [7]:
X_sample = vectorizer.transform([clean_review])
sample_prediction = model.predict(X_sample)[0]

print("Prediction:", sample_prediction)

Prediction: negative


<a id="section5"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">5. Pydantic Input Validation</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">5.1 Why validation is needed</h3>

<p>Incoming requests should never be trusted automatically. Pydantic checks whether the request matches the schema we declared.</p>
<table style="width:100%;border-collapse:collapse;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Request</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Result</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>{"review": "Great movie"}</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Valid</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>{"age": 25}</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Rejected — review is missing</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>{"review": ""}</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Rejected when minimum length is enforced</td></tr>
</table>
</div>

In [8]:
from pydantic import BaseModel, Field

class ReviewInput(BaseModel):
    review: str = Field(
        ...,
        min_length=1,
        description="A movie review to classify as positive or negative."
    )

print("Pydantic request schema created.")

Pydantic request schema created.


<a id="section6"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">6. Create the FastAPI Application</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">6.1 Create app</h3>

<p><code>app = FastAPI()</code> creates the API application object that Uvicorn will run.</p>

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">6.2 Add a simple health route</h3>

<p>A root route is useful to confirm quickly that the API is alive.</p>
</div>

In [9]:
from fastapi import FastAPI

app = FastAPI(
    title="IMDb Sentiment Analysis API",
    description="Predicts whether a movie review is positive or negative.",
    version="1.0.0",
)

@app.get("/")
def root():
    return {
        "status": "ok",
        "message": "IMDb Sentiment Analysis API is running."
    }

print("FastAPI application created.")

FastAPI application created.


<a id="section7"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">7. Build POST /predict</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">7.1 Full serving pipeline</h3>


<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Incoming review
      ↓
Pydantic validation
      ↓
preprocess_to_string()
      ↓
vectorizer.transform()
      ↓
model.predict()
      ↓
JSON response</div>


<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Important:</b> This is where Day 1 pays off: we reuse the same preprocessing and saved TF-IDF object instead of rebuilding them by hand.</div>

</div>

In [10]:
@app.post("/predict")
def predict(data: ReviewInput):
    # 1) Same text preprocessing used during training
    cleaned_review = preprocess_to_string(data.review)

    # 2) Use the saved TF-IDF vectorizer
    X = vectorizer.transform([cleaned_review])

    # 3) Predict with the saved Logistic Regression model
    prediction = model.predict(X)[0]

    # 4) Return JSON-compatible data
    return {
        "review": data.review,
        "cleaned_review": cleaned_review,
        "prediction": str(prediction),
    }

print("POST /predict endpoint created.")

POST /predict endpoint created.


<a id="section8"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">8. Test the API Inside the Notebook</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">8.1 TestClient</h3>
<p>FastAPI's TestClient lets us test the endpoints without starting the external Uvicorn server yet.</p>
</div>

In [11]:
from fastapi.testclient import TestClient

client = TestClient(app)

response = client.get("/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'status': 'ok', 'message': 'IMDb Sentiment Analysis API is running.'}


c:\Users\sadee\anaconda3\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [12]:
positive_payload = {
    "review": "This movie was amazing, emotional, and beautifully made."
}

response = client.post("/predict", json=positive_payload)

print("Positive test status:", response.status_code)
print(response.json())

Positive test status: 200
{'review': 'This movie was amazing, emotional, and beautifully made.', 'cleaned_review': 'movie amazing emotional beautifully make', 'prediction': 'positive'}


In [13]:
negative_payload = {
    "review": "I hated this movie. It was boring and terrible."
}

response = client.post("/predict", json=negative_payload)

print("Negative test status:", response.status_code)
print(response.json())

Negative test status: 200
{'review': 'I hated this movie. It was boring and terrible.', 'cleaned_review': 'hat movie boring terrible', 'prediction': 'negative'}


<a id="section9"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">9. Create main.py</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">9.1 Why main.py?</h3>

<p>The notebook is useful for learning and testing, but Uvicorn normally runs a Python module. We place the API code in <code>main.py</code>.</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Day2.ipynb
   ↓
learn + test

main.py
   ↓
Uvicorn
   ↓
real local API server</div>

</div>

In [14]:
from pathlib import Path

main_py = """from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib

from preprocessing import preprocess_to_string

vectorizer = joblib.load("tfidf_vectorizer.joblib")
model = joblib.load("sentiment_model.joblib")

app = FastAPI(
    title="IMDb Sentiment Analysis API",
    description="Predicts whether a movie review is positive or negative.",
    version="1.0.0",
)

class ReviewInput(BaseModel):
    review: str = Field(
        ...,
        min_length=1,
        description="A movie review to classify as positive or negative."
    )

@app.get("/")
def root():
    return {
        "status": "ok",
        "message": "IMDb Sentiment Analysis API is running."
    }

@app.post("/predict")
def predict(data: ReviewInput):
    cleaned_review = preprocess_to_string(data.review)
    X = vectorizer.transform([cleaned_review])
    prediction = model.predict(X)[0]

    return {
        "review": data.review,
        "cleaned_review": cleaned_review,
        "prediction": str(prediction),
    }
"""

Path("main.py").write_text(main_py, encoding="utf-8")

1011

<a id="section10"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">10. Run the API with Uvicorn</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">10.1 Start the local server</h3>

<p>Open a terminal inside <code>Week 9/Day 2</code> and run:</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">uvicorn main:app --reload</div>

<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Part</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Meaning</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>uvicorn</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Run the web server.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>main</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Use <code>main.py</code>.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>app</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Use the <code>app = FastAPI()</code> object.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>--reload</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Reload automatically after code changes during development.</td></tr>
</table>
</div>

<a id="section11"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">11. Test with FastAPI /docs</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">11.1 Automatic documentation</h3>

<p>When Uvicorn is running, open:</p>

<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">http://127.0.0.1:8000/docs</div>

<p>FastAPI automatically generates an interactive page where we can test each endpoint.</p>

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">11.2 Test /predict</h3>

<ol style="line-height:1.8;">
<li>Open <b>POST /predict</b>.</li>
<li>Click <b>Try it out</b>.</li>
<li>Enter a review.</li>
<li>Click <b>Execute</b>.</li>
<li>Check the status code and JSON response.</li>
</ol>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">{
  "review": "This movie was fantastic and I really enjoyed it."
}</div>
</div>

<a id="section12"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">12. Validate Invalid Input</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">12.1 Invalid request</h3>

<p>Now test a request that does not match the Pydantic schema.</p>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">{
  "age": 25
}</div>
<p>The API should reject it cleanly before the model runs.</p>
</div>

In [15]:
invalid_payload = {
    "age": 25
}

response = client.post("/predict", json=invalid_payload)

print("Status code:", response.status_code)
print("Validation response:")
print(response.json())

Status code: 422
Validation response:
{'detail': [{'type': 'missing', 'loc': ['body', 'review'], 'msg': 'Field required', 'input': {'age': 25}}]}


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;">
<b>Why this is a success:</b> A validation error proves that malformed input is stopped before it reaches the preprocessing and prediction pipeline.
</div>

<a id="section13"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">13. Day 2 Definition of Done</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<table style="width:100%;border-collapse:collapse;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Requirement</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Done when...</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Saved artifacts</td><td style="padding:9px;border:1px solid #CBD5E1;">Model, TF-IDF vectorizer, and preprocessing load without retraining.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Pydantic schema</td><td style="padding:9px;border:1px solid #CBD5E1;">The API requires a valid <code>review</code> string.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">POST /predict</td><td style="padding:9px;border:1px solid #CBD5E1;">A review is preprocessed, transformed, predicted, and returned as JSON.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Local server</td><td style="padding:9px;border:1px solid #CBD5E1;">Uvicorn starts successfully.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">/docs test</td><td style="padding:9px;border:1px solid #CBD5E1;">Positive, negative, and invalid requests are tested.</td></tr>
</table>
<div style="font-size:1.08em;font-weight:bold;text-align:center;background-color:#FFFBEB;border:1px solid #FBBF24;padding:14px;border-radius:8px;margin:16px 0;color:#111827;">Final Day 2 flow: request → validation → preprocessing → TF-IDF → model → JSON response.</div>
</div>

<a id="section14"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">14. What I Learned Today</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<table style="width:100%;border-collapse:collapse;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Concept</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Meaning in our project</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>FastAPI</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Serve the saved sentiment model as a REST API.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>POST /predict</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Receive a movie review and return its predicted sentiment.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>Pydantic</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Validate incoming request data before prediction.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>Uvicorn</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Run the FastAPI application as a local web server.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>/docs</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Interactively test endpoints in the browser.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>Consistent preprocessing</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Reuse the same cleaning and saved TF-IDF transformation used during training.</td></tr>
</table>
<div style="font-size:1.08em;font-weight:bold;text-align:center;background-color:#FFFBEB;border:1px solid #FBBF24;padding:14px;border-radius:8px;margin:16px 0;color:#111827;">Day 1 saved the model. Day 2 serves the model.</div>
</div>

<div style="text-align:center; margin-top:28px; color:#475569;">
<b>End of Week 9 — Day 2: Serving the Model with FastAPI</b><br>
Saved deployment artifacts → REST API → Validated prediction endpoint
</div>